# FY2026 Literacy PSR — Aggregate Bassa, Bong & Montserrado

**Inputs (place all four files next to this notebook):**
- `LiteracyBassa.xlsx`
- `LiteracyBong.xlsx`
- `LiteracyMonsteratoResponse.xlsx` (filename kept exactly as supplied)
- `Liberia_PSR_Literacy_FY26(1).xlsx` — **the only reference for objectives, descriptions, targets and PSR layout**

**Output:** `Liberia_PSR_Literacy_Bassa_Bong_Montserrado_FY26_Aggregated.xlsx`

## Aggregation policy

- Uses **quarterly summary reached/achieved values**, not activity-by-activity sums (which could double-count people).
- For each volunteer, indicator, sex and age row, uses the **latest quarter containing a numeric count**, including an explicit `0`.
- Keeps missing cells missing (never interprets blank or `UNVERIFIED` as zero). Retains staff-verification notes for review; numeric counts are **not automatically discarded** because the status is unverified.
- Sums the three **distinct volunteer reports** by the exact indicator/sex/age breakdown in the supplied **Literacy** PSR.
- Keeps **all Literacy PSR objectives, indicator definitions, FY indicator targets (column J), targets for reporting PCVs (column M), and original report title**. The original PSR says 10 volunteers assigned to the FY; that remains unchanged because only three source files are supplied.
- Updates aggregation fields (G, H, I, K, L), using formulas for totals and percentages. For indicators with no records in the three files, the source-based counts are left blank, instead of copying national values.
- Ignores source `GEN_*` sheets because they are **not indicators in the Literacy PSR**; records those omissions in an `Aggregation Notes` tab. Does not use the Math & Science PSR or the STEM source workbooks in any way.
- Adds `Source Breakdown`, `Reference Comparison`, and `Aggregation Notes` worksheets.

**Requires Python 3.10+ and `openpyxl`.** Install in your notebook environment with `%pip install openpyxl` if needed; restart the kernel after installation.

In [1]:
from pathlib import Path
from collections import defaultdict
from copy import copy
import re

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.workbook.properties import CalcProperties

In [2]:
# Change INPUT_DIR if your Excel files are in another folder.
INPUT_DIR = Path('.')
SOURCE_FILES = [
    INPUT_DIR / 'data/redataatindividualpcvlevel/LiteracyBassa.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/LiteracyBong.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/LiteracyMonsteratoResponse.xlsx',
]
REFERENCE_FILE = INPUT_DIR / 'data/projectstatusreports/Liberia_PSR_Literacy_FY26.xlsx'
OUTPUT_FILE = INPUT_DIR / 'Liberia_PSR_Literacy_Bassa_Bong_Montserrado_FY26_Aggregated.xlsx'

QUARTER_STRATEGY = 'latest_nonblank'

for path in [*SOURCE_FILES, REFERENCE_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required file: {path.resolve()}')
if OUTPUT_FILE.resolve() == REFERENCE_FILE.resolve():
    raise ValueError('Output file must differ from the Literacy PSR reference file.')

print('Literacy reference:', REFERENCE_FILE.resolve())
print('Source files:', *(path.name for path in SOURCE_FILES), sep='\n  ')

Literacy reference: /Users/ajgrover/Documents/Scripting/liberia-annual-reporting-26/data/projectstatusreports/Liberia_PSR_Literacy_FY26.xlsx
Source files:
  LiteracyBassa.xlsx
  LiteracyBong.xlsx
  LiteracyMonsteratoResponse.xlsx


In [3]:
def numeric(value):
    """Accept numeric Excel values / numbers as strings; blanks are not zero."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return value
    if isinstance(value, str):
        stripped = value.strip().replace(',', '')
        if stripped:
            try:
                return float(stripped) if any(c in stripped.lower() for c in '.e') else int(stripped)
            except ValueError:
                return None
    return None


def quarter_snapshot(worksheet, excel_row, indices):
    """Retrieve the last numeric quarterly summary, rather than sum activities."""
    observations = [
        (q, numeric(worksheet.cell(excel_row, col).value))
        for q, col in sorted(indices.items())
    ]
    available = [(q, value) for q, value in observations if value is not None]
    if QUARTER_STRATEGY != 'latest_nonblank':
        raise ValueError('Unsupported QUARTER_STRATEGY: ' + QUARTER_STRATEGY)
    if not available:
        return None, ''
    q, value = available[-1]
    return value, f'Q{q}'


def extract_psr_blocks(worksheet):
    """Find all indicator blocks from the Literacy PSR, including its empty ones."""
    blocks = []
    category = None
    objective = None
    current = None
    for excel_row in range(4, worksheet.max_row + 1):
        cell_objective = worksheet.cell(excel_row, 1).value
        if isinstance(cell_objective, str) and cell_objective.strip():
            objective = cell_objective.strip()
        cell_category = str(worksheet.cell(excel_row, 2).value or '').strip()
        if cell_category in ('Output', 'Outcome'):
            category = cell_category
        indicator_label = str(worksheet.cell(excel_row, 3).value or '').strip()
        match = re.match(r'^(ED_(?:LIT|MS)_\d+(?:_[A-Z])?):', indicator_label)
        if match:
            if current is not None:
                raise ValueError(f'No Total row found for {current["code"]}')
            current = {
                'code': match.group(1),
                'description': indicator_label,
                'category': category,
                'objective': objective,
                'first': excel_row,
            }
        label = str(worksheet.cell(excel_row, 4).value or '').strip()
        if label == 'Total':
            if current is None:
                raise ValueError(f'Unassociated Total at row {excel_row}')
            current['last'] = excel_row - 1
            current['total'] = excel_row
            current['reference_reached'] = numeric(worksheet.cell(excel_row, 7).value)
            current['reference_achieved'] = numeric(worksheet.cell(excel_row, 8).value)
            current['fy_target'] = worksheet.cell(excel_row, 10).value
            current['reporter_target'] = worksheet.cell(excel_row, 13).value
            blocks.append(current)
            current = None
    if current is not None:
        raise ValueError(f'Missing Total at end of PSR for {current["code"]}')
    if not blocks:
        raise ValueError('No indicator blocks identified in Literacy PSR.')
    codes = [block['code'] for block in blocks]
    if len(codes) != len(set(codes)):
        raise ValueError('Duplicate indicator codes in Literacy PSR.')
    return blocks


def read_volunteer_workbook(path, valid_indicators, category_by_indicator):
    """Read quarterly summary counts, preserving per-row verification status."""
    workbook = load_workbook(path, read_only=False, data_only=True)
    details = []
    reporters = defaultdict(set)
    skipped = []
    try:
        for worksheet in workbook.worksheets:
            indicator = worksheet.title.strip()
            if indicator not in valid_indicators:
                skipped.append((path.name, indicator, 'Not present in the Literacy PSR'))
                continue
            indicator_label = str(worksheet['B4'].value or '').strip()
            if not indicator_label.startswith(indicator + ':'):
                raise ValueError(f'Indicator mismatch: {path.name} / {indicator}: {indicator_label}')
            kind = 'Outcome' if 'Outcome Indicator' in str(worksheet['A4'].value) else 'Output'
            if kind != category_by_indicator[indicator]:
                raise ValueError(f'Output/Outcome category mismatch: {path.name} / {indicator}')
            volunteer = str(worksheet['B3'].value or '').split(' (')[0].strip()
            reporters[indicator].add((path.name, volunteer))

            reached_indices = {}
            achieved_indices = {}
            verification_col = None
            for col in range(1, worksheet.max_column + 1):
                header = str(worksheet.cell(7, col).value or '').strip()
                match = re.fullmatch(r'(Reached|Achieved)\s+Q([1-4])', header, flags=re.I)
                if match:
                    dest = reached_indices if match.group(1).lower() == 'reached' else achieved_indices
                    dest[int(match.group(2))] = col
                elif 'Latest Q Staff Verified' in header:
                    verification_col = col
            if not reached_indices:
                raise ValueError(f'Quarterly reached headers missing: {path.name} / {indicator}')
            if kind == 'Outcome' and not achieved_indices:
                raise ValueError(f'Quarterly achieved headers missing: {path.name} / {indicator}')

            current_sex = ''
            for excel_row in range(8, min(worksheet.max_row, 500) + 1):
                col_a = str(worksheet.cell(excel_row, 1).value or '').strip()
                col_b = str(worksheet.cell(excel_row, 2).value or '').strip()
                if col_a.lower() == 'key:':
                    break
                if col_a.startswith('Sex:'):
                    current_sex = col_a
                if not current_sex:
                    continue
                # Sex-only rows are valid for teacher indicators. Age is blank for them.
                if col_b and not col_b.startswith('Age:'):
                    continue
                if not col_a.startswith('Sex:') and not col_b.startswith('Age:'):
                    continue
                age = col_b if col_b.startswith('Age:') else ''
                reached, reached_q = quarter_snapshot(worksheet, excel_row, reached_indices)
                achieved, achieved_q = quarter_snapshot(worksheet, excel_row, achieved_indices)
                verification = str(worksheet.cell(excel_row, verification_col).value or '').strip() if verification_col else ''
                details.append({
                    'Source': path.name,
                    'Volunteer': volunteer,
                    'Indicator': indicator,
                    'Sex': current_sex,
                    'Age': age,
                    'Reached': reached,
                    'Achieved': achieved,
                    'Reached quarter': reached_q,
                    'Achieved quarter': achieved_q,
                    'Verification': verification,
                    'Source sheet': worksheet.title,
                    'Excel row': excel_row,
                    'Raw reached': {q: numeric(worksheet.cell(excel_row, col).value) for q, col in reached_indices.items()},
                    'Raw achieved': {q: numeric(worksheet.cell(excel_row, col).value) for q, col in achieved_indices.items()},
                })
    finally:
        workbook.close()
    return details, reporters, skipped

In [4]:
# Read the Literacy reference first; it decides which indicators to aggregate.
report_wb = load_workbook(REFERENCE_FILE)
report_ws = report_wb.worksheets[0]
blocks = extract_psr_blocks(report_ws)
block_map = {block['code']: block for block in blocks}

# Snapshot columns that are explicitly defined by the Literacy PSR.
PSR_PRESERVED_COLS = (1, 2, 3, 4, 5, 6, 10, 13)  # A-F, J, M
preserved_values = {
    (row, col): report_ws.cell(row, col).value
    for row in range(1, report_ws.max_row + 1)
    for col in PSR_PRESERVED_COLS
}

all_details = []
reporters = defaultdict(set)
skipped_sheets = []
for path in SOURCE_FILES:
    details, source_reporters, skipped = read_volunteer_workbook(
        path, set(block_map), {code: block['category'] for code, block in block_map.items()}
    )
    all_details.extend(details)
    skipped_sheets.extend(skipped)
    for code, identities in source_reporters.items():
        reporters[code].update(identities)
if not all_details:
    raise ValueError('No Literacy rows were read from the supplied sources.')

print(f'Literacy PSR indicators: {len(blocks)}')
print(f'Source disaggregation records: {len(all_details)}')
print(f'Source indicator sheets: {sum(len(v) for v in reporters.values())}')
print(f'Unique volunteers: {len({(d["Source"], d["Volunteer"]) for d in all_details})}')
print('Skipped non-PSR sheets:', skipped_sheets)

Literacy PSR indicators: 25
Source disaggregation records: 109
Source indicator sheets: 19
Unique volunteers: 3
Skipped non-PSR sheets: [('LiteracyBong.xlsx', 'GEN_001', 'Not present in the Literacy PSR'), ('LiteracyBong.xlsx', 'GEN_002', 'Not present in the Literacy PSR')]


In [5]:
# Aggregate by the Literacy PSR's exact indicator/sex/age groups.
agg = defaultdict(lambda: {'Reached': 0, 'Achieved': 0, 'has_reached': False, 'has_achieved': False})
for record in all_details:
    key = (record['Indicator'], record['Sex'], record['Age'])
    entry = agg[key]
    for measure in ('Reached', 'Achieved'):
        value = record[measure]
        if value is not None:
            if value < 0:
                raise ValueError(f'Negative {measure} in {record["Source"]} / {key}')
            entry[measure] += value
            entry['has_' + measure.lower()] = True

# Ensure that every counted disaggregation is represented in the Literacy PSR.
valid_breakdowns = {
    (b['code'], str(report_ws.cell(row, 4).value or '').strip(),
     str(report_ws.cell(row, 5).value or '').strip())
    for b in blocks for row in range(b['first'], b['last'] + 1)
}
unmatched = sorted(set(agg) - valid_breakdowns)
if unmatched:
    raise ValueError('Source disaggregations absent from Literacy PSR: ' + repr(unmatched))

summary = {}
for block in blocks:
    code = block['code']
    rows = [v for (ind, _sex, _age), v in agg.items() if ind == code]
    reached_exists = any(v['has_reached'] for v in rows)
    achieved_exists = any(v['has_achieved'] for v in rows)
    summary[code] = {
        'reached': sum(v['Reached'] for v in rows if v['has_reached']) if reached_exists else None,
        'achieved': sum(v['Achieved'] for v in rows if v['has_achieved']) if achieved_exists else None,
        'pcvs': len(reporters[code]),
    }
for code, counts in summary.items():
    if counts['pcvs']:
        print(f'{code:<14} | PCVs={counts["pcvs"]} | reached={counts["reached"]} | achieved={counts["achieved"]}')

ED_LIT_002     | PCVs=2 | reached=20 | achieved=None
ED_LIT_003     | PCVs=1 | reached=6 | achieved=None
ED_LIT_030     | PCVs=2 | reached=20 | achieved=None
ED_LIT_032     | PCVs=1 | reached=2 | achieved=None
ED_LIT_006     | PCVs=2 | reached=17 | achieved=14
ED_LIT_013     | PCVs=2 | reached=8 | achieved=14
ED_LIT_016     | PCVs=3 | reached=1070 | achieved=None
ED_LIT_017     | PCVs=2 | reached=86 | achieved=None
ED_LIT_018     | PCVs=1 | reached=45 | achieved=None
ED_LIT_019     | PCVs=3 | reached=442 | achieved=257


In [6]:
# Update only aggregation fields in a *copy* of the uploaded Literacy PSR.
# All objectives / descriptions / targets originate from the Literacy PSR workbook.
for block in blocks:
    for row in range(block['first'], block['total'] + 1):
        for col in (7, 8, 9, 11, 12):  # G / H / I / K / L
            report_ws.cell(row, col).value = None

    code = block['code']
    for row in range(block['first'], block['last'] + 1):
        sex = str(report_ws.cell(row, 4).value or '').strip()
        age = str(report_ws.cell(row, 5).value or '').strip()
        counts = agg.get((code, sex, age))
        if counts is None:
            continue
        if counts['has_reached']:
            report_ws.cell(row, 7).value = counts['Reached']
        if block['category'] == 'Outcome' and counts['has_achieved']:
            report_ws.cell(row, 8).value = counts['Achieved']
        if block['category'] == 'Outcome':
            report_ws.cell(row, 9).value = f'=IF(OR(G{row}="",H{row}=""),"",IFERROR(H{row}/G{row},""))'
            report_ws.cell(row, 9).number_format = '0.0%'

    first, last, total = block['first'], block['last'], block['total']
    report_ws.cell(total, 7).value = f'=IF(COUNT(G{first}:G{last})=0,"",SUM(G{first}:G{last}))'
    report_ws.cell(total, 12).value = len(reporters[code])
    if block['category'] == 'Outcome':
        report_ws.cell(total, 8).value = f'=IF(COUNT(H{first}:H{last})=0,"",SUM(H{first}:H{last}))'
        report_ws.cell(total, 9).value = f'=IF(OR(G{total}="",H{total}=""),"",IFERROR(H{total}/G{total},""))'
        numerator = f'H{total}'
    else:
        report_ws.cell(total, 8).value = 'N/A'
        report_ws.cell(total, 9).value = 'N/A'
        numerator = f'G{total}'
    # Target J and target PCV count M are *never* changed.
    report_ws.cell(total, 11).value = f'=IF(J{total}>0,IF({numerator}="",0,{numerator}/J{total}),"")'
    report_ws.cell(total, 7).number_format = '#,##0'
    report_ws.cell(total, 9).number_format = '0.0%' if block['category'] == 'Outcome' else 'General'
    report_ws.cell(total, 11).number_format = '0.0%'

# Keep the original Literacy PSR heading and the original 10-volunteer statement.
for (row, col), original in preserved_values.items():
    assert report_ws.cell(row, col).value == original, f'Protected Literacy PSR field changed: {row},{col}'

In [7]:
def style_auxiliary_sheet(ws, widths, last_col):
    for row_number, fill_color, font_size in [(1, '123B4A', 14), (4, '175A6D', 10)]:
        for c in range(1, last_col + 1):
            cell = ws.cell(row_number, c)
            cell.fill = PatternFill('solid', fgColor=fill_color)
            cell.font = Font(bold=True, color='FFFFFF', size=font_size)
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        ws.row_dimensions[row_number].height = 33 if row_number == 1 else 42
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    ws.freeze_panes = 'F5'
    ws.sheet_view.showGridLines = False


def write_table(ws, header, records):
    for col, title in enumerate(header, start=1):
        ws.cell(4, col).value = title
    for row, record in enumerate(records, start=5):
        for col, value in enumerate(record, start=1):
            if value is not None:
                ws.cell(row, col).value = value
    if records:
        ws.auto_filter.ref = f'A4:{get_column_letter(len(header))}{4+len(records)}'


audit = report_wb.create_sheet('Source Breakdown')
audit['A1'] = 'LITERACY FY26  |  SOURCE BREAKDOWN'
audit['A2'] = ('3 volunteer workbooks; latest numeric quarterly summary per row. '
               'Blank means no numeric value; review UNVERIFIED flags manually.')
audit_headers = [
    'Workbook', 'Volunteer', 'Indicator', 'Sex', 'Age', 'Reached', 'Achieved',
    'Reached quarter', 'Achieved quarter', 'Staff verification', 'Source sheet', 'Source row',
    'Reached Q1', 'Reached Q2', 'Reached Q3', 'Reached Q4',
    'Achieved Q1', 'Achieved Q2', 'Achieved Q3', 'Achieved Q4',
]
audit_data = []
for record in all_details:
    audit_data.append([
        record['Source'], record['Volunteer'], record['Indicator'], record['Sex'], record['Age'],
        record['Reached'], record['Achieved'], record['Reached quarter'], record['Achieved quarter'],
        record['Verification'], record['Source sheet'], record['Excel row'],
        *[record['Raw reached'].get(q) for q in (1, 2, 3, 4)],
        *[record['Raw achieved'].get(q) for q in (1, 2, 3, 4)],
    ])
write_table(audit, audit_headers, audit_data)
style_auxiliary_sheet(audit, {
    'A':30, 'B':25, 'C':17, 'D':17, 'E':16, 'F':13, 'G':13,
    'H':18, 'I':18, 'J':21, 'K':18, 'L':13,
    **{get_column_letter(c):14 for c in range(13, 21)}
}, 20)
for i, data in enumerate(audit_data, start=5):
    if 'UNVERIFIED' in str(data[9]).upper():
        audit.cell(i, 10).font = Font(bold=True, color='B91C1C')
    for col in (6, 7, *range(13, 21)):
        audit.cell(i, col).number_format = '#,##0'

In [8]:
comparison = report_wb.create_sheet('Reference Comparison')
comparison['A1'] = 'LITERACY FY26 | AGGREGATED vs ORIGINAL PSR'
comparison['A2'] = ('Original reference counts are shown for comparison only. The original '
                    'literacy PSR may include volunteer activity not represented in the 3 uploads.')
comparison_headers = [
    'Indicator', 'Type', 'Reporting PCVs in files',
    'Aggregated reached', 'Original PSR reached', 'Reached difference',
    'Aggregated achieved', 'Original PSR achieved', 'Achieved difference',
    'Literacy FY target', 'Target reporting PCVs', 'Indicator definition',
]
comparison_data = []
for block in blocks:
    code = block['code']
    reached, achieved = summary[code]['reached'], summary[code]['achieved']
    ref_reached, ref_achieved = block['reference_reached'], block['reference_achieved']
    comparison_data.append([
        code, block['category'], summary[code]['pcvs'],
        reached, ref_reached,
        reached - ref_reached if reached is not None and ref_reached is not None else None,
        achieved if block['category'] == 'Outcome' else None,
        ref_achieved if block['category'] == 'Outcome' else None,
        achieved - ref_achieved if achieved is not None and ref_achieved is not None else None,
        block['fy_target'], block['reporter_target'], block['description'],
    ])
write_table(comparison, comparison_headers, comparison_data)
style_auxiliary_sheet(comparison, {
    'A':18, 'B':15, 'C':23, 'D':21, 'E':23, 'F':20,
    'G':21, 'H':24, 'I':21, 'J':19, 'K':23, 'L':62,
}, 12)
comparison.freeze_panes = 'D5'
for row_number in range(5, 5 + len(comparison_data)):
    for col in range(3, 12):
        comparison.cell(row_number, col).number_format = '#,##0.##'

In [9]:
notes = report_wb.create_sheet('Aggregation Notes')
notes['A1'] = 'FY2026 LITERACY AGGREGATION — METHOD & SCOPE'
notes['A2'] = 'Only the uploaded Literacy PSR is authoritative for objectives, indicators, FY targets and PSR layout.'
notes_rows = [
    ('Reference workbook', REFERENCE_FILE.name),
    ('Volunteer files', ', '.join(path.name for path in SOURCE_FILES)),
    ('Reporting volunteers in supplied files', len({(d['Source'], d['Volunteer']) for d in all_details})),
    ('Quarter selection', 'Latest quarter with a numeric summary cell (including zero); do not sum activity cells'),
    ('Verified status policy', 'Status retained for audit; numerical counts are not dropped automatically for UNVERIFIED labels'),
    ('Missing count policy', 'Blank is missing, not zero'),
    ('Protected PSR fields', 'A-F (including objectives and indicator descriptions), J (FY targets), M (target number of PCVs)'),
    ('Aggregation fields updated', 'G (reached), H (achieved), I (% achieved), K (% of FY target), L (PCVs reporting)'),
    ('Original volunteer assignment', str(report_ws['A2'].value or '')),
    ('No-report indicators', ', '.join(b['code'] for b in blocks if not reporters[b['code']]) or 'None'),
    ('Source sheets excluded', ', '.join(f'{p} / {s}' for p, s, _ in skipped_sheets) or 'None'),
    ('Source count caveat', 'Files are not evidence that all volunteers assigned to the project have reported'),
]
for row_number, pair in enumerate(notes_rows, start=4):
    for col, val in enumerate(pair, start=1):
        notes.cell(row_number, col).value = val
notes.column_dimensions['A'].width = 36
notes.column_dimensions['B'].width = 112
notes.sheet_view.showGridLines = False
notes.freeze_panes = 'B4'
for c in range(1, 3):
    cell = notes.cell(1, c)
    cell.fill = PatternFill('solid', fgColor='123B4A')
    cell.font = Font(bold=True, color='FFFFFF', size=14)
notes.row_dimensions[1].height = 34
for row_number in range(4, 4 + len(notes_rows)):
    notes.cell(row_number, 1).font = Font(bold=True, color='175A6D')
    notes.cell(row_number, 2).alignment = Alignment(wrap_text=True, vertical='top')
    notes.row_dimensions[row_number].height = 35 if row_number in (5, 10, 12, 14) else 29

# Encourage Excel to calculate totals, % values, and comparison formulas on opening.
report_wb.calculation = CalcProperties(calcMode='auto', fullCalcOnLoad=True, forceFullCalc=True)

# Final verification: ensure all Literacy PSR objectives, target cells and descriptions are unchanged.
assert all(report_ws.cell(r, c).value == val for (r, c), val in preserved_values.items()), \
    'A protected Literacy PSR field changed unexpectedly.'
assert len([r for r in all_details if r['Indicator'].startswith('ED_LIT_')]) == len(all_details)
assert all(record['Indicator'] in block_map for record in all_details)

report_wb.save(OUTPUT_FILE)
report_wb.close()

# Reopen to test actual .xlsx output structure and protected fields.
check_wb = load_workbook(OUTPUT_FILE, data_only=False, read_only=True)
check_ws = check_wb.worksheets[0]
for (r, c), original in preserved_values.items():
    assert check_ws.cell(r, c).value == original or (original in ('', None) and check_ws.cell(r, c).value in ('', None)), \
        f'Protected Literacy PSR value changed after saving: {r},{c}'
for b in blocks:
    assert check_ws.cell(b['total'], 12).value == summary[b['code']]['pcvs']
    assert check_ws.cell(b['total'], 10).value == b['fy_target']
    assert check_ws.cell(b['total'], 13).value == b['reporter_target']
assert {'Source Breakdown', 'Reference Comparison', 'Aggregation Notes'} <= set(check_wb.sheetnames)
check_wb.close()

print('\nSUCCESS: saved and verified', OUTPUT_FILE.resolve())
print('Worksheets: Literacy PSR, Source Breakdown, Reference Comparison, Aggregation Notes')
print('Note: open in Excel to recalculate formula-based total and percent cells.')


SUCCESS: saved and verified /Users/ajgrover/Documents/Scripting/liberia-annual-reporting-26/Liberia_PSR_Literacy_Bassa_Bong_Montserrado_FY26_Aggregated.xlsx
Worksheets: Literacy PSR, Source Breakdown, Reference Comparison, Aggregation Notes
Note: open in Excel to recalculate formula-based total and percent cells.
